<a href="https://colab.research.google.com/github/MXC66ai/MultimodalLLM-ObjectDetection/blob/main/MedVitEncoder_decoder_medmnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install medmnist



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.5 MB/s eta 0:00:00


In [2]:
from medmnist import INFO, Evaluator
from medmnist import OrganAMNIST, OrganMNIST3D

In [3]:
from torch.utils.data import Dataset, DataLoader

导入依赖

多模态医学图像 transforms（2D + 3D 专用）

加载 MedMNIST（2D + 3D）

构建多模态融合数据集 MultiModalMedMNIST

构建 DataLoader

你的融合模型（保持不变，除输入尺寸外无需改）

训练循环（Dice / IoU / Loss）


In [4]:
import torch.nn as nn


In [5]:
class Med2DTransforms:
    def __init__(self):
        # PIL-based transforms
        self.aug_pil = T.Compose([
            T.RandomRotation(15),
            T.RandomHorizontalFlip(),
        ])

        # tensor transform
        self.to_tensor = T.ToTensor()

    def windowing_np(self, img_np, WL=40, WW=200):
        # windowing on numpy array
        img_np = np.clip((img_np - (WL - WW/2)) / WW, 0, 1)
        return img_np

    def __call__(self, img):
        # img is PIL.Image
        # Step 1: convert to numpy
        img_np = np.array(img).astype(np.float32)

        # Step 2: windowing
        img_np = self.windowing_np(img_np)

        # Step 3: numpy -> PIL
        img_pil = T.functional.to_pil_image((img_np * 255).astype(np.uint8))

        # Step 4: PIL transforms
        img_pil = self.aug_pil(img_pil)

        # Step 5: convert to tensor
        img_tensor = self.to_tensor(img_pil)

        return img_tensor


class Med3DTransforms:
    def __init__(self):
        pass

    def random_flip(self, img):
        if random.random() < 0.5:
            img = np.flip(img, axis=2).copy()
        return img

    def __call__(self, img):
        # img shape = (D, H, W) numpy

        img = img.astype(np.float32)

        # normalize
        if img.max() > 0:
            img = img / img.max()

        img = self.random_flip(img)

        # 关键修复：确保永远只变成 [1, D, H, W]
        if img.ndim == 3:
            img = torch.tensor(img).unsqueeze(0)  # [1,D,H,W]
        elif img.ndim == 4:
            # 如果已经是 [C,D,H,W]，则不再 unsqueeze
            img = torch.tensor(img)
        else:
            raise ValueError("Unexpected 3D image shape:", img.shape)

        return img.float()


In [6]:
info2d = INFO["organamnist"]
info3d = INFO["organmnist3d"]

DataClass2D = getattr(__import__("medmnist"), info2d["python_class"])
DataClass3D = getattr(__import__("medmnist"), info3d["python_class"])


In [7]:
class MultiModalMedMNIST(Dataset):
    def __init__(self, split='train'):
        self.data2d = DataClass2D(split=split, transform=Med2DTransforms(), download=True)
        self.data3d = DataClass3D(split=split, transform=Med3DTransforms(), download=True)

        self.len2d = len(self.data2d)
        self.len3d = len(self.data3d)
        self.length = self.len3d

    def __len__(self):
        return self.length

    def __getitem__(self, idx):

        # 3D data
        img3d, label3d = self.data3d[idx]
        label3d = int(label3d[0])      # ❗ 修复

        # 找匹配的 2D data
        match_found = False
        while not match_found:
            idx2d = random.randint(0, self.len2d - 1)
            img2d, label2d = self.data2d[idx2d]
            label2d = int(label2d[0])  # ❗ 修复
            if label2d == label3d:
                match_found = True

        # 伪 segmentation mask：shape 必须对齐
        target = (img2d > 0.5).float()

        return img2d, img3d, target


In [8]:
import torchvision.transforms as T
train_loader = DataLoader(
    MultiModalMedMNIST("train"),
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    MultiModalMedMNIST("test"),
    batch_size=32,
    shuffle=False
)


100%|██████████| 38.2M/38.2M [00:56<00:00, 677kB/s]
100%|██████████| 32.7M/32.7M [00:50<00:00, 643kB/s]


In [9]:
# 2D Encoder
class Encoder2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 28 → 14

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14 → 7
        )

    def forward(self, x):
        return self.enc(x)  # [B,64,7,7]


# 3D Encoder
class Encoder3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv3d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool3d(2),
        )

    def forward(self, x):
        x = self.enc(x)  # [B,32,D/4,H/4,W/4] => [B,32,7,7,7]
        x = x.mean(dim=2)  # depth squeeze：→ [B,32,7,7]
        return x


# Fusion + Transformer
class FusionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.fuse = nn.Conv2d(64 + 32, 64, 1)

        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=64,
                nhead=8,
                batch_first=True,
            ),
            num_layers=2
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),  # 7→14
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 2, stride=2),   # 14→28
            nn.Sigmoid()
        )

    def forward(self, f2d, f3d):
        B, C, H, W = f2d.shape

        fused = torch.cat([f2d, f3d], dim=1)
        fused = self.fuse(fused)

        tokens = fused.flatten(2).permute(0, 2, 1)  # [B,49,64]
        tokens = self.transformer(tokens)
        fused = tokens.permute(0, 2, 1).reshape(B, 64, H, W)

        out = fused
        return self.decoder(out)  # [B,1,28,28]


# Total fusion model
class FusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc2d = Encoder2D()
        self.enc3d = Encoder3D()
        self.fusion = FusionTransformer()

    def forward(self, img2d, img3d):
        f2d = self.enc2d(img2d)
        f3d = self.enc3d(img3d)
        return self.fusion(f2d, f3d)


In [10]:
train_dataset = MultiModalMedMNIST("train")
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)


In [14]:
import numpy as np
import torch

import random

In [15]:
img2d, img3d, target = next(iter(train_loader))
print(img3d.shape)


torch.Size([32, 1, 28, 28, 28])


In [16]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FusionModel().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def dice(pred, target):
    pred = (pred > 0.5).float()
    return (2 * (pred*target).sum()) / (pred.sum() + target.sum() + 1e-8)

def iou(pred, target):
    pred = (pred > 0.5).float()
    inter = (pred*target).sum()
    union = pred.sum() + target.sum() - inter
    return inter / (union + 1e-8)


for epoch in range(3):
    model.train()
    losses, dices, ious = [], [], []

    for img2d, img3d, target in train_loader:
        img2d, img3d, target = img2d.to(device), img3d.to(device), target.to(device)

        optimizer.zero_grad()
        out = model(img2d, img3d)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        dices.append(dice(out, target).item())
        ious.append(iou(out, target).item())

    print(f"Epoch {epoch+1}: Loss={np.mean(losses):.4f}, Dice={np.mean(dices):.4f}, IoU={np.mean(ious):.4f}")


Epoch 1: Loss=0.4944, Dice=0.8969, IoU=0.8141
Epoch 2: Loss=0.3887, Dice=0.9218, IoU=0.8554
Epoch 3: Loss=0.2856, Dice=0.9508, IoU=0.9063
